In [8]:
import numpy as np

# Data

In [11]:
X = np.array([[0, 0],
              [0, 1],
              [1, 0],
              [1, 1]])

In [7]:
y = np.array([[0],
              [1],
              [1],
              [0]])

# Exercise

For the supplied XOR dataset, implement a simple feedforward neural network with one hidden layer with 2 units using sigmoid activation and 1 output unit with sigmoid activation. Train it using the MSE loss function and stochastic gradient descent (SGD), initializing the weights to small random values. Use a learning rate of 0.1, batch size of 2, and train for 10 epochs. After training, report the final weights and the predicted outputs for the XOR inputs.

In [ ]:
# Sigmoid activation and its derivative
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_deriv(a):
    return a * (1 - a)

# Reproducibility
np.random.seed(42)

# Network dimensions
input_size  = 2
hidden_size = 2
output_size = 1

# Weight initialisation (small random values)
W1 = np.random.randn(input_size, hidden_size) * 0.1   # (2, 2)
b1 = np.zeros((1, hidden_size))                        # (1, 2)
W2 = np.random.randn(hidden_size, output_size) * 0.1  # (2, 1)
b2 = np.zeros((1, output_size))                        # (1, 1)

# Hyper-parameters
lr         = 0.1
batch_size = 2
epochs     = 10
n_samples  = X.shape[0]

# Training loop
for epoch in range(1, epochs + 1):
    # Shuffle data each epoch
    idx = np.random.permutation(n_samples)
    X_s, y_s = X[idx], y[idx]

    epoch_loss = 0.0
    for start in range(0, n_samples, batch_size):
        Xb = X_s[start:start + batch_size]  # (batch, 2)
        yb = y_s[start:start + batch_size]  # (batch, 1)
        m  = Xb.shape[0]

        # --- Forward pass ---
        z1 = Xb @ W1 + b1          # (m, 2)
        a1 = sigmoid(z1)           # (m, 2)
        z2 = a1 @ W2 + b2          # (m, 1)
        a2 = sigmoid(z2)           # (m, 1)

        # MSE loss
        loss = np.mean((a2 - yb) ** 2)
        epoch_loss += loss

        # --- Backward pass ---
        # Output layer
        dL_da2 = 2 * (a2 - yb) / m          # (m, 1)
        dL_dz2 = dL_da2 * sigmoid_deriv(a2) # (m, 1)
        dL_dW2 = a1.T @ dL_dz2              # (2, 1)
        dL_db2 = dL_dz2.sum(axis=0, keepdims=True)  # (1, 1)

        # Hidden layer
        dL_da1 = dL_dz2 @ W2.T              # (m, 2)
        dL_dz1 = dL_da1 * sigmoid_deriv(a1) # (m, 2)
        dL_dW1 = Xb.T @ dL_dz1             # (2, 2)
        dL_db1 = dL_dz1.sum(axis=0, keepdims=True)  # (1, 2)

        # --- SGD update ---
        W2 -= lr * dL_dW2
        b2 -= lr * dL_db2
        W1 -= lr * dL_dW1
        b1 -= lr * dL_db1

    print(f"Epoch {epoch:2d} | loss: {epoch_loss / (n_samples // batch_size):.6f}")

# --- Final report ---
print("\n=== Final Weights ===")
print(f"W1:\n{W1}")
print(f"b1: {b1}")
print(f"W2:\n{W2}")
print(f"b2: {b2}")

print("\n=== Predictions vs Ground Truth ===")
a1_final = sigmoid(X @ W1 + b1)
preds    = sigmoid(a1_final @ W2 + b2)
for xi, yi, pi in zip(X, y, preds):
    print(f"  Input: {xi}  Target: {yi[0]}  Predicted: {pi[0]:.4f}")